<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# Module 2.7: Enums, Decoders and Bundles
**Prev: [Testing with ChiselSim](2.6_chiselsim.ipynb)**<br>
**Next: [Memories](2.8_memories.ipynb)**

## Motivation
A processor is full of small encodings: the state of a controller, the operation an ALU performs,
the fields of an instruction word. Writing them as bare `UInt` constants works, but the compiler then
cannot tell an ALU operation from a register number. This module shows the Chisel tools that give
these encodings names and types:

- `ChiselEnum` for named states and operation codes, and `switch`/`is` to branch on them;
- `BitPat` to match bit patterns with don't-care bits, with `ListLookup` and `decoder`;
- `MixedVec` for a vector whose elements have different types;
- the connection operators `:<>=`, `:<=` and `:>=` for bundles with signals in both directions.

## Setup
If a cell fails or hangs, see [Troubleshooting](docs/troubleshooting.md).

In [ ]:
interp.load.module(os.pwd / "source" / "load-ivy.sc")

In [ ]:
import chisel3._
import chisel3.Input // avoid a name clash with almond.input.Input
import chisel3.util._
import chisel3.simulator.PeekPokeAPI._

---
# ChiselEnum

A `ChiselEnum` is a hardware type with a fixed set of named values. Each `Value` gets the next
encoding, starting at 0, and the width is the smallest that holds them all:

In [ ]:
object Color extends ChiselEnum {
  val Red, Green, Blue = Value
}

println(Color.all.map(c => s"${c} = ${c.litValue}").mkString(", "))
println(s"width: ${Color.getWidth} bits")

You can also choose the encodings, for example to match a specification. They must be literals in
increasing order. A value declared without an encoding takes the next one:

In [ ]:
object Opcode extends ChiselEnum {
  val Load   = Value("b0000011".U)
  val Fence  = Value("b0001111".U)
  val OpImm  = Value("b0010011".U)
  val Auipc  = Value("b0010111".U)
  val Store  = Value("b0100011".U)
  val Op     = Value("b0110011".U)
  val Lui    = Value("b0110111".U)
  val Branch = Value("b1100011".U)
  val Jalr   = Value("b1100111".U)
  val Jal    = Value("b1101111".U)
  val System = Value("b1110011".U)
}
println(s"width: ${Opcode.getWidth} bits")

These are the 11 major opcodes of RV32I. An enum is used like any other Chisel type: `Opcode()`
makes a port or wire of that type, and the values are compared with `===`. Converting between an
enum and its encoding is explicit: `.asUInt` gives the encoding, and `Opcode(x)` turns a `UInt`
back into an `Opcode`.

That last conversion is risky when `x` can hold an encoding that is not a member: `Opcode` uses
7 bits, but only 11 of the 128 values are opcodes. Chisel warns at elaboration time about such a
cast (and rejects a `UInt` wider than the enum). `Opcode.safe(x)` is the cast without the warning:
it returns the enum together with a `Bool` that says whether `x` is a valid member, so the design
handles the other values explicitly (here, with an illegal-instruction flag):

In [ ]:
class OpcodeCheck extends Module {
  val io = IO(new Bundle {
    val bits    = Input(UInt(7.W))
    val isStore = Output(Bool())
    val illegal = Output(Bool())
  })
  val (opcode, valid) = Opcode.safe(io.bits)
  io.isStore := valid && opcode === Opcode.Store
  io.illegal := !valid
}

simulate(new OpcodeCheck) { c =>
  c.io.bits.poke("b0100011".U)
  c.io.isStore.expect(true.B)
  c.io.illegal.expect(false.B)
  c.io.bits.poke("b1111111".U)
  c.io.illegal.expect(true.B)
}
println("SUCCESS!!")

---
# switch and is

`switch`/`is` is a `when` chain that compares one signal against literal values. It reads well with
enums, so it is the usual way to write a state machine. Here is a controller that waits for a request,
spends two cycles working, and then reports that it is done:

In [ ]:
object State extends ChiselEnum {
  val Idle, Busy1, Busy2, Done = Value
}

class Controller extends Module {
  val io = IO(new Bundle {
    val start = Input(Bool())
    val done  = Output(Bool())
  })
  val state = RegInit(State.Idle)
  switch(state) {
    is(State.Idle) { when(io.start) { state := State.Busy1 } }
    is(State.Busy1) { state := State.Busy2 }
    is(State.Busy2) { state := State.Done }
    is(State.Done) { state := State.Idle }
  }
  io.done := state === State.Done
}

println(getVerilog(new Controller))

In the SystemVerilog, `state` is a plain 2-bit register and the states are the constants 0 to 3:
enums cost nothing in hardware. The names live in your Chisel code, where the compiler checks that a
`State` is never compared with, say, an `Opcode`.

<span style="color:red">**Exercise: Traffic Light**</span><br>
Complete the traffic light below. It shows `Green` for 3 cycles, then `Yellow` for 1 cycle, then
`Red` for 2 cycles, and starts again with `Green`. The counter `timer` counts down; when it reaches 0,
move to the next state and load `timer` with the duration of that state minus one.

In [ ]:
object Light extends ChiselEnum {
  val Green, Yellow, Red = Value
}

class TrafficLight extends Module {
  val io = IO(new Bundle {
    val light = Output(Light())
  })
  val state = RegInit(Light.Green)
  val timer = RegInit(2.U(2.W)) // Green lasts 3 cycles: 2, 1, 0
  io.light := state

  when(timer =/= 0.U) {
    timer := timer - 1.U
  }.otherwise {
    ???
  }
}

In [ ]:
simulate(new TrafficLight) { c =>
  val expected = Seq.fill(3)(Light.Green) ++ Seq(Light.Yellow) ++ Seq.fill(2)(Light.Red)
  for (cycle <- 0 until 12) {
    c.io.light.expect(expected(cycle % expected.length))
    c.clock.step()
  }
}
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>Solution</strong></label>
<article>
<pre style="background-color:#f7f7f7">
    switch(state) {
      is(Light.Green)  { state := Light.Yellow; timer := 0.U }
      is(Light.Yellow) { state := Light.Red;    timer := 1.U }
      is(Light.Red)    { state := Light.Green;  timer := 2.U }
    }
</pre></article></div></section></div>

---
# BitPat: matching with don't-care bits

A `BitPat` is a bit pattern in which `?` matches either value. It is compared with a `UInt` using `===`.
RISC-V instructions are a good example: the `ADDI` instruction is any 32-bit word whose `funct3` field
(bits 14 to 12) is `000` and whose opcode (bits 6 to 0) is `0010011`; the other bits are operands.

In [ ]:
val ADDI = BitPat("b?????????????????000?????0010011")
val ADD  = BitPat("b0000000??????????000?????0110011")

class IsAddi extends Module {
  val io = IO(new Bundle {
    val inst = Input(UInt(32.W))
    val addi = Output(Bool())
  })
  io.addi := ADDI === io.inst
}

simulate(new IsAddi) { c =>
  c.io.inst.poke("h00500093".U) // addi x1, x0, 5
  c.io.addi.expect(true.B)
  c.io.inst.poke("h002081b3".U) // add x3, x1, x2
  c.io.addi.expect(false.B)
}
println("SUCCESS!!")

`ListLookup(key, default, mapping)` matches a key against a list of `BitPat`s and returns the list of
signals of the first match (or `default`). Each entry of the list is one control signal, so one table
describes a whole decoder:

In [ ]:
class TinyDecoder extends Module {
  val io = IO(new Bundle {
    val inst     = Input(UInt(32.W))
    val regWrite = Output(Bool())
    val useImm   = Output(Bool())
  })
  // each list is (regWrite, useImm)
  val signals = ListLookup(
    io.inst,
    List(false.B, false.B),
    Array(
      ADDI -> List(true.B, true.B),
      ADD  -> List(true.B, false.B)
    )
  )
  io.regWrite := signals(0)
  io.useImm   := signals(1)
}
println(getVerilog(new TinyDecoder))

Why not `MuxLookup`? `MuxLookup(key, default)(Seq(k1 -> v1, ...))` compares `key === k` exactly, for
keys that are hardware `UInt`s (or values of a `ChiselEnum`), so every bit takes part
in the comparison. A `BitPat` is not a hardware value: it describes a set of values, so it cannot be a
`MuxLookup` key. Use `MuxLookup` for exact keys and `ListLookup` (or `decoder`, below) for patterns.

`chisel3.util.experimental.decode.decoder` goes one step further: it takes a whole truth table of
input and output `BitPat`s and minimizes the logic. `decoder.qmc` uses the Quine-McCluskey minimizer
that is built into Chisel (`decoder(...)` tries the external `espresso` tool first, and logs an error
and falls back to Quine-McCluskey when it is not installed). Here the outputs are the two signals of `TinyDecoder`, packed into 2 bits:

In [ ]:
import chisel3.util.experimental.decode.{decoder, TruthTable}

class TableDecoder extends Module {
  val io = IO(new Bundle {
    val inst     = Input(UInt(32.W))
    val regWrite = Output(Bool())
    val useImm   = Output(Bool())
  })
  val table = TruthTable(
    Seq(
      ADDI -> BitPat("b11"),
      ADD  -> BitPat("b10")
    ),
    default = BitPat("b00")
  )
  val out = decoder.qmc(io.inst, table)
  io.regWrite := out(1)
  io.useImm   := out(0)
}

simulate(new TableDecoder) { c =>
  c.io.inst.poke("h00500093".U) // addi
  c.io.regWrite.expect(true.B)
  c.io.useImm.expect(true.B)
  c.io.inst.poke("h002081b3".U) // add
  c.io.regWrite.expect(true.B)
  c.io.useImm.expect(false.B)
  c.io.inst.poke("h00000000".U) // not in the table
  c.io.regWrite.expect(false.B)
}
println("SUCCESS!!")

<span style="color:red">**Exercise: Instruction Format**</span><br>
Every RISC-V instruction has one of six formats (R, I, S, B, U and J) that says where its operands and
its immediate are. The format depends only on the opcode, the low 7 bits of the instruction:

| Opcode (bits 6-0) | Instructions | Format |
| :--- | :--- | :--- |
| `0110011` | register-register (`add`, ...) | R |
| `0010011`, `0000011`, `1100111`, `0001111`, `1110011` | register-immediate (`addi`, ...), loads, `jalr`, `fence`, `ecall`/`ebreak` | I |
| `0100011` | stores | S |
| `1100011` | branches | B |
| `0110111`, `0010111` | `lui`, `auipc` | U |
| `1101111` | `jal` | J |

Complete `FormatDecoder` with `ListLookup` (or `decoder.qmc`) so that `format` is the format of `inst`,
and `valid` is false for every other opcode (then `format` does not matter). The helper `op` builds a
`BitPat` that matches any instruction with the given opcode.

In [ ]:
object Format extends ChiselEnum {
  val R, I, S, B, U, J = Value
}

class FormatDecoder extends Module {
  val io = IO(new Bundle {
    val inst   = Input(UInt(32.W))
    val format = Output(Format())
    val valid  = Output(Bool())
  })
  def op(bits: String) = BitPat("b" + "?" * 25 + bits) // matches any instruction with this opcode

  ???
}

In [ ]:
simulate(new FormatDecoder) { c =>
  val cases = Seq(
    "h002081b3" -> Format.R, // add  x3, x1, x2
    "h00500093" -> Format.I, // addi x1, x0, 5
    "h0040a103" -> Format.I, // lw   x2, 4(x1)
    "h000080e7" -> Format.I, // jalr x1, 0(x1)
    "h0ff0000f" -> Format.I, // fence
    "h00000073" -> Format.I, // ecall
    "h0020a223" -> Format.S, // sw   x2, 4(x1)
    "h00208463" -> Format.B, // beq  x1, x2, 8
    "h000010b7" -> Format.U, // lui  x1, 1
    "h00001097" -> Format.U, // auipc x1, 1
    "h008000ef" -> Format.J  // jal  x1, 8
  )
  for ((inst, format) <- cases) {
    c.io.inst.poke(inst.U)
    c.io.valid.expect(true.B, s"instruction $inst")
    c.io.format.expect(format, s"instruction $inst")
  }
  c.io.inst.poke("h0000000b".U) // custom-0 opcode: not RV32I
  c.io.valid.expect(false.B)
}
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-2" />
<label for="check-2"><strong>Solution</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  val signals = ListLookup(io.inst, List(false.B, Format.R), Array(
    op("0110011") -&gt; List(true.B, Format.R),
    op("0010011") -&gt; List(true.B, Format.I),
    op("0000011") -&gt; List(true.B, Format.I),
    op("1100111") -&gt; List(true.B, Format.I),
    op("0001111") -&gt; List(true.B, Format.I),
    op("1110011") -&gt; List(true.B, Format.I),
    op("0100011") -&gt; List(true.B, Format.S),
    op("1100011") -&gt; List(true.B, Format.B),
    op("0110111") -&gt; List(true.B, Format.U),
    op("0010111") -&gt; List(true.B, Format.U),
    op("1101111") -&gt; List(true.B, Format.J)
  ))
  io.valid  := signals(0)
  io.format := signals(1)
</pre></article></div></section></div>

---
# MixedVec

A `Vec` holds elements of one type. When a generator needs a vector whose elements have different
widths or even different types, use `MixedVec`. Its elements are indexed with Scala integers
(`v(0)`, `v(1)`), not with hardware signals, because there is no single type a dynamic index could return.

In [ ]:
class Widths extends Module {
  val io = IO(new Bundle {
    val out = Output(MixedVec(Seq(UInt(4.W), UInt(8.W), Bool())))
  })
  io.out(0) := 15.U
  io.out(1) := 255.U
  io.out(2) := true.B
}
println(getVerilog(new Widths))

<span style="color:red">**Exercise: Field Splitter**</span><br>
Write a generator that splits a word into fields of the given widths, starting at the least significant
bit. For `widths = Seq(7, 5, 3)` and a 15-bit input, `fields(0)` is bits 6-0, `fields(1)` is bits 11-7
and `fields(2)` is bits 14-12 (these are the `opcode`, `rd` and `funct3` fields of a RISC-V instruction).

In [ ]:
class FieldSplitter(widths: Seq[Int]) extends Module {
  val io = IO(new Bundle {
    val in     = Input(UInt(widths.sum.W))
    val fields = Output(MixedVec(widths.map(w => UInt(w.W))))
  })
  ???
}

In [ ]:
def checkSplitter(widths: Seq[Int]): Unit = simulate(new FieldSplitter(widths)) { c =>
  val rnd     = new scala.util.Random(0)
  val offsets = widths.scanLeft(0)(_ + _)
  for (_ <- 0 until 50) {
    val word = BigInt(widths.sum, rnd)
    c.io.in.poke(word.U)
    for (i <- widths.indices) {
      val field = (word >> offsets(i)) & ((BigInt(1) << widths(i)) - 1)
      c.io.fields(i).expect(field.U, s"widths $widths, field $i")
    }
  }
}
checkSplitter(Seq(7, 5, 3))
checkSplitter(Seq(1, 4, 8, 3))
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-3" />
<label for="check-3"><strong>Solution</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  // the offset of each field is the sum of the widths before it
  val offsets = widths.scanLeft(0)(_ + _)
  for (i &lt;- widths.indices) {
    io.fields(i) := io.in(offsets(i) + widths(i) - 1, offsets(i))
  }
</pre></article></div></section></div>

---
# Connecting bundles with signals in both directions

A ready-valid interface (`Decoupled`) has signals that flow forward (`valid`, `bits`) and one that
flows backward (`ready`). Seen from the bundle named on one side of a connection (say `io.out`), the
fields that point the same way as that bundle are *aligned*, and the ones under a `Flipped` are
*flipped*. Chisel 7 has a family of connection operators that say which of them to connect, always
written `consumer <op> producer`:

| Operator | Connects | Typical use |
| :--- | :--- | :--- |
| `c :<>= p` | aligned fields from `p` to `c`, and flipped fields from `c` to `p` | pass an interface through |
| `c :<= p` | only the aligned fields, from `p` to `c` | the forward half |
| `c :>= p` | only the flipped fields, from `c` to `p` | the backward half |

`:<>=` is exactly `:<=` and `:>=` together. Splitting it lets you insert logic in one direction. Below,
a `Gate` passes a stream through but only lets data through while `enable` is set: the forward
signals are connected with `:<=` and then `valid` is overridden (last connect wins); `ready` is driven
by hand instead of `:>=`.

In [ ]:
class Gate extends Module {
  val io = IO(new Bundle {
    val enable = Input(Bool())
    val in     = Flipped(Decoupled(UInt(8.W)))
    val out    = Decoupled(UInt(8.W))
  })
  io.out :<= io.in // valid and bits forward
  io.out.valid := io.in.valid && io.enable // then override valid
  io.in.ready  := io.out.ready && io.enable
}

simulate(new Gate) { c =>
  c.io.in.valid.poke(true.B)
  c.io.in.bits.poke(42.U)
  c.io.out.ready.poke(true.B)
  c.io.enable.poke(false.B)
  c.io.out.valid.expect(false.B)
  c.io.in.ready.expect(false.B)
  c.io.enable.poke(true.B)
  c.io.out.valid.expect(true.B)
  c.io.out.bits.expect(42.U)
  c.io.in.ready.expect(true.B)
}
println("SUCCESS!!")

When nothing needs to be inserted, `:<>=` passes the whole interface through in one line, and the
SystemVerilog shows both directions connected:

In [ ]:
class PassThrough extends Module {
  val io = IO(new Bundle {
    val in  = Flipped(Decoupled(UInt(8.W)))
    val out = Decoupled(UInt(8.W))
  })
  io.out :<>= io.in
}
println(getVerilog(new PassThrough))

The older `<>` also connects both directions, but it decides which side drives each field from where
each side lives (a port of this module, a port of a child, or a wire). The new operators use only the
types, so ports and wires behave the same.

They also check more: both sides must have the same fields, flipped the same way, and a connection
that could truncate a value is an error (mark the producer with `.squeeze` if you mean it).

---
# You're done!

[Return to the top.](#top)